# Sustained Attention Benchmark

By Conrad Kleykamp

Evaluates whether models can track and aggregate specific targets accurately across a long sequential context, without skipping entries, approximating, or losing count midway. Runs across all models available in the Kaggle environment (`kbench.llms`). A fixed judge model (`kbench.judge_llm`) scores all responses to ensure consistent cross-model evaluation.

Each task row contains 5 judge-evaluated criteria plus 1 regex hard-check on the final numeric answer. Total assertions per model: 30 (5 rows × 6 criteria).

> **Environment note:** Designed to run inside a Kaggle notebook where `kbench.llm`, `kbench.judge_llm`, and `kbench.llms` are auto-configured.

In [4]:
import kaggle_benchmarks as kbench
import pandas as pd

In [5]:
TASK_DATA = pd.DataFrame([
    {
        "context": (
            "Transaction 1: Alice sent $200 to Bob. "
            "Transaction 2: Carol bought groceries for $45. "
            "Transaction 3: Bob transferred $80 to Dave. "
            "Transaction 4: Eve paid $120 for utilities. "
            "Transaction 5: Alice received $300 from Frank. "
            "Transaction 6: Dave spent $60 at a restaurant. "
            "Transaction 7: Carol received $150 from Grace. "
            "Transaction 8: Frank paid $90 for insurance. "
            "Transaction 9: Bob bought electronics for $400. "
            "Transaction 10: Eve received $250 from Alice. "
            "Transaction 11: Grace paid $30 for a subscription. "
            "Transaction 12: Dave received $110 from Carol. "
            "Transaction 13: Alice paid $75 for phone bills. "
            "Transaction 14: Frank spent $200 on travel. "
            "Transaction 15: Bob received $95 from Eve."
        ),
        "question": "What is Alice's net balance across all transactions? Show your working.",
        "expected_answer": "-.*225|negative.*225",
        "criteria": [
            "Response accounts for Alice sending $200 (debit).",
            "Response accounts for Alice receiving $300 from Frank (credit).",
            "Response accounts for Alice paying $75 for phone bills (debit).",
            "Response accounts for Eve receiving $250 from Alice (debit).",
            "Final net balance calculated is negative $225.",
        ],
    },
    {
        "context": (
            "Parcel depot log (12 days, 3 couriers): "
            "Day 1: Lee received 12 parcels. Kim received 8 parcels. "
            "Day 2: Lee dispatched 3 parcels. Park received 5 parcels. "
            "Day 3: Lee received 8 parcels. Kim dispatched 4 parcels. Park received 10 parcels. "
            "Day 4: Kim received 6 parcels. Park dispatched 2 parcels. "
            "Day 5: Lee received 15 parcels. Park received 8 parcels. "
            "Day 6: Lee dispatched 5 parcels. Kim received 11 parcels. "
            "Day 7: Kim dispatched 3 parcels. Park received 7 parcels. "
            "Day 8: Lee received 18 parcels. Park dispatched 6 parcels. "
            "Day 9: Lee dispatched 6 parcels. Kim received 5 parcels. "
            "Day 10: Lee received 14 parcels. Kim dispatched 8 parcels. Park received 12 parcels. "
            "Day 11: Park dispatched 4 parcels. Kim received 9 parcels. "
            "Day 12: Lee dispatched 9 parcels. Park received 3 parcels."
        ),
        "question": (
            "Tracking Lee's parcel count only: what is Lee's final parcel count at the end of Day 12, "
            "and on which day did Lee's running count first exceed 50? "
            "Show your running balance for each day Lee had activity."
        ),
        "expected_answer": "44",
        "criteria": [
            "Response correctly identifies all of Lee's activity: received on Days 1, 3, 5, 8, 10 and dispatched on Days 2, 6, 9, 12.",
            "Response correctly tracks Lee's running balance reaching 53 on Day 10.",
            "Response correctly identifies Day 10 as the first day Lee's count exceeded 50.",
            "Response correctly calculates Lee's final parcel count as 44 after the Day 12 dispatch.",
            "Response does not include Kim's or Park's parcel counts in Lee's total.",
        ],
    },
    {
        "context": (
            "Monthly payroll register (20 entries): "
            "Entry 1: Alice, Engineering, Regular pay $4200. "
            "Entry 2: Bob, Marketing, Regular pay $3800. "
            "Entry 3: Carol, Engineering, Regular pay $5100. "
            "Entry 4: Dave, Operations, Regular pay $3500. "
            "Entry 5: Eve, Marketing, Regular pay $4600. "
            "Entry 6: Alice, Engineering, Performance bonus +$800. "
            "Entry 7: Frank, Operations, Regular pay $3200. "
            "Entry 8: Bob, Marketing, Travel reimbursement +$350. "
            "Entry 9: Carol, Engineering, Tax adjustment -$620. "
            "Entry 10: Grace, Engineering, Regular pay $4800. "
            "Entry 11: Dave, Operations, Overtime bonus +$400. "
            "Entry 12: Eve, Marketing, Performance bonus +$500. "
            "Entry 13: Alice, Engineering, Expense deduction -$240. "
            "Entry 14: Frank, Operations, Pay correction +$180. "
            "Entry 15: Grace, Engineering, Training deduction -$150. "
            "Entry 16: Bob, Marketing, Pay correction +$200. "
            "Entry 17: Carol, Engineering, Performance bonus +$750. "
            "Entry 18: Dave, Operations, Tax adjustment -$280. "
            "Entry 19: Alice, Engineering, Overtime bonus +$320. "
            "Entry 20: Grace, Engineering, Performance bonus +$600."
        ),
        "question": (
            "What is the total net payroll for the Engineering department only, "
            "including all regular pay, bonuses, and deductions? Show your working per employee."
        ),
        "expected_answer": "15[,]?560",
        "criteria": [
            "Response correctly identifies Alice, Carol, and Grace as the Engineering employees and excludes Marketing and Operations entries.",
            "Response correctly calculates Alice's net Engineering pay as $5080 (4200 + 800 - 240 + 320).",
            "Response correctly calculates Carol's net Engineering pay as $5230 (5100 - 620 + 750).",
            "Response correctly calculates Grace's net Engineering pay as $5250 (4800 - 150 + 600).",
            "Total Engineering net payroll calculated is $15560.",
        ],
    },
    {
        "context": (
            "Weekly expense report: "
            "Entry 1: Category=Travel, Amount=$120, Submitted by Lee. "
            "Entry 2: Category=Software, Amount=$45, Submitted by Kim. "
            "Entry 3: Category=Travel, Amount=$85, Submitted by Kim. "
            "Entry 4: Category=Office Supplies, Amount=$30, Submitted by Lee. "
            "Entry 5: Category=Software, Amount=$200, Submitted by Lee. "
            "Entry 6: Category=Travel, Amount=$310, Submitted by Park. "
            "Entry 7: Category=Office Supplies, Amount=$55, Submitted by Kim. "
            "Entry 8: Category=Travel, Amount=$95, Submitted by Lee."
        ),
        "question": "What is the total amount spent on Travel expenses across all employees?",
        "expected_answer": "610",
        "criteria": [
            "Response identifies Entry 1 ($120) as a Travel expense.",
            "Response identifies Entry 3 ($85) as a Travel expense.",
            "Response identifies Entry 6 ($310) as a Travel expense.",
            "Response identifies Entry 8 ($95) as a Travel expense.",
            "Response correctly calculates total Travel expenses as $610.",
        ],
    },
    {
        "context": (
            "Customer support ticket log (25 tickets): "
            "Ticket 1: Morgan, Technical, Resolved, 3.0 hours. "
            "Ticket 2: Riley, Billing, Resolved, 2.5 hours. "
            "Ticket 3: Taylor, General, Resolved, 1.5 hours. "
            "Ticket 4: Morgan, Billing, Resolved, 5.0 hours. "
            "Ticket 5: Riley, Technical, Unresolved. "
            "Ticket 6: Taylor, Refund, Resolved, 4.0 hours. "
            "Ticket 7: Morgan, General, Resolved, 4.0 hours. "
            "Ticket 8: Riley, General, Resolved, 2.0 hours. "
            "Ticket 9: Morgan, Technical, Unresolved. "
            "Ticket 10: Taylor, Billing, Resolved, 3.5 hours. "
            "Ticket 11: Riley, Refund, Resolved, 3.0 hours. "
            "Ticket 12: Morgan, Billing, Resolved, 3.5 hours. "
            "Ticket 13: Riley, Technical, Resolved, 1.5 hours. "
            "Ticket 14: Taylor, General, Unresolved. "
            "Ticket 15: Morgan, Refund, Resolved, 5.5 hours. "
            "Ticket 16: Riley, Billing, Unresolved. "
            "Ticket 17: Taylor, Technical, Resolved, 2.5 hours. "
            "Ticket 18: Morgan, General, Resolved, 6.0 hours. "
            "Ticket 19: Riley, General, Resolved, 3.5 hours. "
            "Ticket 20: Morgan, Technical, Unresolved. "
            "Ticket 21: Taylor, Billing, Resolved, 2.0 hours. "
            "Ticket 22: Morgan, Billing, Resolved, 5.0 hours. "
            "Ticket 23: Riley, Refund, Resolved, 4.0 hours. "
            "Ticket 24: Morgan, Refund, Resolved, 4.0 hours. "
            "Ticket 25: Taylor, Technical, Resolved, 3.0 hours."
        ),
        "question": (
            "For agent Morgan only: what is the average resolution time in hours across resolved tickets, "
            "and what percentage of Morgan's tickets are unresolved? Show your working."
        ),
        "expected_answer": "4\\.5",
        "criteria": [
            "Response correctly identifies all 10 of Morgan's tickets (1, 4, 7, 9, 12, 15, 18, 20, 22, 24) from the 25-entry log.",
            "Response correctly identifies 8 resolved and 2 unresolved tickets for Morgan.",
            "Response correctly sums Morgan's resolved ticket resolution times as 36.0 hours.",
            "Response correctly calculates Morgan's average resolution time as 4.5 hours.",
            "Response correctly calculates that 20% of Morgan's tickets are unresolved.",
        ],
    },
])


@kbench.task(name="sustained_attention", store_task=False)
def sustained_attention(
    llm,
    context: str,
    question: str,
    expected_answer: str,
    criteria: list,
):
    """
    Evaluating sustained attention: model must track and aggregate specific
    targets accurately across a long sequential context. Judge is a fixed
    external model. Each row has 1 regex hard-check plus 5 judge criteria.
    """
    prompt = (
        f"Read the following records carefully and answer the question. "
        f"Track every relevant entry -- do not skip or approximate.\n\n"
        f"Records:\n{context}\n\n"
        f"Question: {question}"
    )

    response = llm.prompt(prompt)

    kbench.assertions.assert_contains_regex(
        pattern=expected_answer,
        text=response,
        expectation="Response should contain the correct numeric answer",
    )

    report = kbench.assertions.assess_response_with_judge(
        criteria=criteria,
        response_text=response,
        judge_llm=kbench.judge_llm,
    )
    for result in report.results:
        kbench.assertions.assert_true(
            result.passed,
            expectation=result.criterion,
        )


sustained_attention.bind_dataframe(TASK_DATA, name="sustained_attention", store_task=True)

Task(func=<function Task.bind_dataframe.<locals>.func at 0x79b6c59a6700>, name='sustained_attention', description='Evaluating sustained attention: model must track and aggregate specific\ntargets accurately across a long sequential context. Judge is a fixed\nexternal model. Each row has 1 regex hard-check plus 5 judge criteria.', result_type=<class 'kaggle_benchmarks.results.PassCount'>, version=1, store_task=True, store_run=True)

In [6]:
all_runs = []

for model_name, model_llm in kbench.llms.items():
    print(f"\n--- {model_name} ---")
    for i, row in TASK_DATA.iterrows():
        try:
            run = sustained_attention.run(
                llm=model_llm,
                context=row["context"],
                question=row["question"],
                expected_answer=row["expected_answer"],
                criteria=row["criteria"],
            )
            all_runs.append((model_name, i, run))
            passed = sum(ar.passed for ar in run.assertion_results)
            total = len(run.assertion_results)
            print(f"  Row {i}: {run.status.name} -- {passed}/{total} assertions passed")
        except Exception as e:
            print(f"  Row {i}: ERROR -- {e}")


--- anthropic/claude-haiku-4-5@20251001 ---
  Row 0: SUCCESS -- 3/6 assertions passed
  Row 1: SUCCESS -- 6/6 assertions passed
  Row 2: SUCCESS -- 6/6 assertions passed
  Row 3: SUCCESS -- 6/6 assertions passed
  Row 4: SUCCESS -- 6/6 assertions passed

--- anthropic/claude-opus-4-1@20250805 ---
  Row 0: ERROR -- 'NoneType' object is not subscriptable
  Row 1: ERROR -- 'NoneType' object is not subscriptable
  Row 2: ERROR -- 'NoneType' object is not subscriptable
  Row 3: ERROR -- 'NoneType' object is not subscriptable
  Row 4: ERROR -- 'NoneType' object is not subscriptable

--- anthropic/claude-opus-4-5@20251101 ---
  Row 0: SUCCESS -- 6/6 assertions passed
  Row 1: SUCCESS -- 6/6 assertions passed
  Row 2: SUCCESS -- 6/6 assertions passed
  Row 3: SUCCESS -- 6/6 assertions passed
  Row 4: SUCCESS -- 6/6 assertions passed

--- anthropic/claude-opus-4-6@default ---
  Row 0: SUCCESS -- 6/6 assertions passed
  Row 1: SUCCESS -- 6/6 assertions passed
  Row 2: SUCCESS -- 6/6 assertions 

In [7]:
def run_to_record(model_name, row_index, run):
    passed = sum(ar.passed for ar in run.assertion_results)
    total = len(run.assertion_results)
    assertion_summary = "; ".join(
        f"{'PASS' if ar.passed else 'FAIL'}: {ar.expectation}"
        for ar in run.assertion_results
    )
    return {
        "model": model_name,
        "row": row_index,
        "status": run.status.name,
        "assertions_passed": passed,
        "assertions_total": total,
        "pass_rate": round(passed / total, 2) if total > 0 else None,
        "error_message": run.error_message,
        "assertion_details": assertion_summary,
    }


records = [run_to_record(model_name, i, run) for model_name, i, run in all_runs]
results_df = pd.DataFrame(records)
pd.set_option("display.max_colwidth", 100)

print("\n=== Per-row results ===")
display(results_df)

print("\n=== Summary by model ===")
summary = (
    results_df.groupby("model")
    .agg(
        assertions_passed=("assertions_passed", "sum"),
        assertions_total=("assertions_total", "sum"),
    )
    .assign(pass_rate=lambda df: (df["assertions_passed"] / df["assertions_total"]).round(3))
    .sort_values("pass_rate", ascending=False)
    .reset_index()
)
display(summary)


=== Per-row results ===


,model,row,status,assertions_passed,assertions_total,pass_rate,error_message,assertion_details
0,anthropic/claude-haiku-4-5@20251001,0,SUCCESS,3,6,0.5,None,FAIL: Response should contain the correct numeric answer; PASS: Response accounts for Alice send...
1,anthropic/claude-haiku-4-5@20251001,1,SUCCESS,6,6,1.0,None,PASS: Response should contain the correct numeric answer; PASS: Response correctly identifies al...
2,anthropic/claude-haiku-4-5@20251001,2,SUCCESS,6,6,1.0,None,PASS: Response should contain the correct numeric answer; PASS: Response correctly identifies Al...
3,anthropic/claude-haiku-4-5@20251001,3,SUCCESS,6,6,1.0,None,PASS: Response should contain the correct numeric answer; PASS: Response identifies Entry 1 ($12...
4,anthropic/claude-haiku-4-5@20251001,4,SUCCESS,6,6,1.0,None,PASS: Response should contain the correct numeric answer; PASS: Response correctly identifies al...
...,...,...,...,...,...,...,...,...
155,zai/glm-5,0,SUCCESS,6,6,1.0,None,PASS: Response should contain the correct numeric answer; PASS: Response accounts for Alice send...
156,zai/glm-5,1,SUCCESS,6,6,1.0,None,PASS: Response should contain the correct numeric answer; PASS: Response correctly identifies al...
157,zai/glm-5,2,SUCCESS,6,6,1.0,None,PASS: Response should contain the correct numeric answer; PASS: Response correctly identifies Al...
158,zai/glm-5,3,SUCCESS,6,6,1.0,None,PASS: Response should contain the correct numeric answer; PASS: Response identifies Entry 1 ($12...



=== Summary by model ===


,model,assertions_passed,assertions_total,pass_rate
0,anthropic/claude-opus-4-5@20251101,30,30,1.000
1,anthropic/claude-opus-4-6@default,30,30,1.000
2,anthropic/claude-sonnet-4-5@20250929,30,30,1.000
3,anthropic/claude-sonnet-4-6@default,30,30,1.000
4,deepseek-ai/deepseek-v3.1,30,30,1.000
5,deepseek-ai/deepseek-r1-0528,30,30,1.000
6,google/gemini-2.0-flash-lite,30,30,1.000
7,deepseek-ai/deepseek-v3.2,30,30,1.000
8,google/gemini-3.1-pro-preview,30,30,1.000
9,google/gemini-3-flash-preview,30,30,1.000


In [8]:
output_path = "sustained_attention_test_results.csv"
summary.to_csv(output_path, index=False)
print(f"Saved: {output_path}")
display(summary)

detail_path = "sustained_attention_test_detailed_results.csv"
results_df.to_csv(detail_path, index=False)
print(f"Saved detailed results: {detail_path}")

Saved: sustained_attention_test_results.csv


,model,assertions_passed,assertions_total,pass_rate
0,anthropic/claude-opus-4-5@20251101,30,30,1.000
1,anthropic/claude-opus-4-6@default,30,30,1.000
2,anthropic/claude-sonnet-4-5@20250929,30,30,1.000
3,anthropic/claude-sonnet-4-6@default,30,30,1.000
4,deepseek-ai/deepseek-v3.1,30,30,1.000
5,deepseek-ai/deepseek-r1-0528,30,30,1.000
6,google/gemini-2.0-flash-lite,30,30,1.000
7,deepseek-ai/deepseek-v3.2,30,30,1.000
8,google/gemini-3.1-pro-preview,30,30,1.000
9,google/gemini-3-flash-preview,30,30,1.000


Saved detailed results: sustained_attention_test_detailed_results.csv
